# exp/r2-mgan on Colab (T4): D pretrain, then queue [gru_v6m1, gru_v6m2, gru_v6m3]
Runtime -> Change runtime type -> T4 GPU, then Run all. A failed run raises with the config name and stops the queue (no silent skip). After a disconnect, just Run all again: the D pretrain is skipped once `d_pretrain.pt` is finished, each config resumes from its `last.pt` on Drive (`--resume`; G, D, both optimizers, replay buffer, RNG), and a config whose `train_log.csv` already reached the final epoch or has an `early_stop` row is skipped. Each run writes `best.pt` (best validation PESQ, inference weights only), `last.pt` (full state, every epoch, ~130 MB) and `train_log.csv`; console output is appended to `train_stdout.log` there.

MetricGAN+-style fine-tuning (Fu et al. 2021) of gru_v5b: a PESQ discriminator D (4 conv2d + spectral norm, GAP, MLP, sigmoid; 94k params) learns normalised PESQ-WB of (enhanced/noisy/clean vs clean); G adds lambda * (D(enh, clean) - 1)^2. All variants: warm start weights/gru_v5b.pt, magnitude mask, Adam G 1e-4 / D 2e-4 (constant), seed 0, 60 epochs, val PESQ every epoch, early stop after 15 epochs without gain. D is pretrained once for 2 epochs on frozen gru_v5b outputs and shared.

| config | cmag | MR-STFT | -SI-SDR | lambda_metric |
|---|---|---|---|---|
| gru_v6m1 | 1.0 | 0.5 | 0.3 | 0.05 |
| gru_v6m2 | 1.0 | 0.2 | 0 | 0.3 |
| gru_v6m3 | 0.2 | 0 | 0 | 1.0 |

Time: each step computes 32 PESQ-WB scores (16 enhanced + 16 noisy) on Colab's 2 CPUs, so training is PESQ-bound: ~1.4-1.8 s/step, ~2.5-3 min/epoch incl. validation, up to ~3 h per variant (60 epochs), ~5 min for the D pretrain, up to ~9 h for the whole queue (less with early stops). A high-RAM runtime with more CPUs scales the PESQ part down almost linearly (`--pesq-jobs` = os.cpu_count()).

Drive layout: dataset at `MyDrive/noise_reduction_challenge/{train,validation}/...`; checkpoints go to `MyDrive/nrc_ckpt/<config_stem>/`, the shared D to `MyDrive/nrc_ckpt/mgan_d_pretrain/d_pretrain.pt`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
import os
!rm -rf /content/repo && git clone -b exp/r2-mgan --depth 1 https://github.com/pauljh4323/SGSYS_13thTeam_NoiseReductionChallenge.git /content/repo
%cd /content/repo
assert os.path.getsize('weights/gru_v5b.pt') > 1_000_000, 'weights/gru_v5b.pt missing (init_from)'
print('init weights OK:', os.path.getsize('weights/gru_v5b.pt'), 'bytes; CPUs:', os.cpu_count())

In [ ]:
!pip install -q pesq==0.0.4 pystoi soundfile pyyaml

In [ ]:
!mkdir -p /content/data && cp -rn /content/drive/MyDrive/noise_reduction_challenge /content/data/

In [ ]:
import os, yaml, pandas as pd
from pathlib import Path
CONFIGS = ["configs/dl/gru_v6m1.yaml", "configs/dl/gru_v6m2.yaml", "configs/dl/gru_v6m3.yaml"]  # run in this order
DATA = '/content/data/noise_reduction_challenge'
CKPT_ROOT = Path('/content/drive/MyDrive/nrc_ckpt')
D_PRE = CKPT_ROOT / 'mgan_d_pretrain' / 'd_pretrain.pt'
D_PRE.parent.mkdir(parents=True, exist_ok=True)
import subprocess, sys

def run_logged(cmd, log, name):
    """Run cmd, stream its output here and append it to log; raise if it exits non-zero (no silent `| tee`)."""
    print('$ ' + ' '.join(map(str, cmd)), flush=True)
    with open(log, 'a', encoding='utf-8') as f:
        proc = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='', flush=True)
            f.write(line)
        rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f'{name} failed with exit code {rc}; see {log}')

# D pretrain (2 epochs on frozen gru_v5b outputs); train_mgan.py exits at once if D_PRE is already finished
run_logged([sys.executable, 'scripts/train_mgan.py', '--config', CONFIGS[0], '--pretrain-d', D_PRE, '--data-root', DATA,
            '--device', 'auto'], D_PRE.parent / 'train_stdout.log', 'D pretrain')

In [ ]:
def ckpt_dir(cfg): return CKPT_ROOT / Path(cfg).stem

def finished(cfg):
    log = ckpt_dir(cfg) / 'train_log.csv'
    epochs = yaml.safe_load(open(cfg))['train']['epochs']
    if not log.is_file() or os.path.getsize(log) == 0:
        return False
    df = pd.read_csv(log)
    if len(df) == 0:
        return False
    early = 'early_stop' in df and (pd.to_numeric(df.early_stop, errors='coerce') == 1).any()
    return bool(early) or df.epoch.max() >= epochs

for cfg in CONFIGS:
    d = ckpt_dir(cfg)
    if finished(cfg):
        print(f'skip {cfg}: final epoch reached or early-stopped')
        continue
    d.mkdir(parents=True, exist_ok=True)
    run_logged([sys.executable, 'scripts/train_mgan.py', '--config', cfg, '--data-root', DATA, '--ckpt-dir', d,
                '--d-init', D_PRE, '--resume', '--device', 'auto'], d / 'train_stdout.log', Path(cfg).stem)

In [ ]:
for cfg in CONFIGS:
    log = ckpt_dir(cfg) / 'train_log.csv'
    if not log.is_file():
        print(f'{Path(cfg).stem}: no train_log.csv'); continue
    df = pd.read_csv(log).dropna(subset=['val_pesq'])
    if df.empty:
        print(f'{Path(cfg).stem}: no validation yet'); continue
    r = df.loc[df.val_pesq.idxmax()]
    print(f'{Path(cfg).stem}: best epoch {int(r.epoch)}, val PESQ {r.val_pesq:.4f}, val SNR {r.val_snr:.2f} dB '
          f'(last epoch {int(pd.read_csv(log).epoch.max())}, mean D(enh) {r.d_enh:.3f}, train PESQ {r.train_pesq_enh:.3f})')